# El miedo que no baja · reproducción en Google Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rlondono1620-max/el-miedo-que-no-baja/blob/main/notebooks/colab_reproducir.ipynb)

**Bogotá DataJam 2026 · Equipo #16 The data squad** (Brandow León, Samuel Riveros y Rafael Londoño, estudiantes de Ciencia de Datos).

Este cuaderno corre el proyecto completo en la nube, sin instalar nada en el computador:

1. clona el repositorio con los datos incluidos;
2. instala las dependencias de `requirements.txt`;
3. ejecuta `src/pipeline.py`: integración de 18 fuentes, validación del registro, modelos, índice y exportes;
4. muestra los resultados clave y las figuras.

Tiempo aproximado: 8 a 12 minutos (los bootstraps son lo más lento). Las semillas están fijas: dos corridas dan las mismas cifras.
El análisis paso a paso, con cada decisión explicada, está en `notebooks/el_miedo_que_no_baja.ipynb`.

## 1. Clonar el repositorio

In [ ]:
!git clone --depth 1 https://github.com/rlondono1620-max/el-miedo-que-no-baja.git
%cd el-miedo-que-no-baja

## 2. Instalar dependencias

In [ ]:
!pip install -q -r requirements.txt

## 3. Ejecutar el pipeline completo

`src/pipeline.py` hace, en orden: descarga o reutiliza cada fuente con su registro de URL, fecha y sha1; limpia y une por el código oficial de localidad;
valida cada serie contra una fuente independiente (Policía y Medicina Legal); estima tendencias, el modelo inferencial, los modelos de machine learning
y la tipología; calcula la cifra oculta y el Índice de Brecha con bootstrap; y exporta tablas, figuras y los datos del visor.

In [ ]:
!python src/pipeline.py

## 4. Resultados clave

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_colwidth", 120)
R = json.load(open("outputs/resultados.json", encoding="utf-8"))
claves = ["hurto_personas", "hurto_personas_capa", "ciudad_insegura_pct", "zona_insegura_pct", "cifra_oculta_bogota",
          "ibs_vs_cifra_oculta", "ibs_vs_cifra_oculta_parcial", "victimas_vs_registro", "ibs_top5", "n_bienal"]
pd.Series({k: R[k] for k in claves}, name="valor").to_frame()

## 5. Validación del registro (H0)

Razón entre la capa abierta de la Secretaría de Seguridad y el registro de la Policía (SIEDCO), por año. Hasta 2022 la capa contaba entre 2,3 y 2,6
veces los hurtos de la Policía; desde 2023 coinciden caso por caso. La caída de la serie abierta estaba en el método, no en la calle.

In [ ]:
pd.read_csv("outputs/tablas/registro_razon_scj_siedco.csv")

In [ ]:
display(Image("outputs/figuras/fig1_serie_delitos.png"))

## 6. Qué explica el miedo (H1)

Logit ponderado por el factor de expansión de la Bienal 2025, con efectos marginales promedio e IC por bootstrap de conglomerados.
La prueba del delito registrado usa wild cluster bootstrap con pesos de Webb, porque con 19 localidades los errores agrupados son optimistas.

In [ ]:
pd.read_csv("outputs/tablas/m_hechos.csv")

In [ ]:
pd.read_csv("outputs/tablas/lpm_wild.csv")

In [ ]:
display(Image("outputs/figuras/fig3_efectos_marginales.png"))

## 7. Machine learning

Regresión logística, bosque aleatorio y gradient boosting con validación cruzada 5 × 5 y AUC ponderada fuera de muestra.
La prueba de contexto compara el delito registrado con saber la localidad (efectos fijos).

In [ ]:
pd.read_csv("outputs/tablas/ml_comparacion_modelos_sin_calificaciones.csv")

In [ ]:
pd.read_csv("outputs/tablas/ml_contexto_localidad.csv")

## 8. Validaciones territoriales y la hipótesis descartada (H2)

El Índice de Brecha y la cifra oculta se correlacionan (ρ ≈ 0,72), pero los dos usan el delito registrado. Con Spearman parcial, controlando
ese componente, la correlación desaparece: la relación era mecánica y H2 no se sostiene.

In [ ]:
pd.read_csv("outputs/tablas/validaciones.csv")

In [ ]:
pd.read_csv("outputs/tablas/validacion_parcial.csv")

## 9. Cifra oculta e Índice de Brecha de Seguridad

In [ ]:
pd.read_csv("outputs/tablas/cifra_oculta.csv").sort_values("razon", ascending=False).head(10)

In [ ]:
pd.read_csv("outputs/tablas/ibs.csv").head(10)

In [ ]:
display(Image("outputs/figuras/fig7_indice_brecha.png"))

## 10. Enfoque de género (H3)

In [ ]:
pd.read_csv("outputs/tablas/emv_teme_acoso_sexual.csv")

In [ ]:
display(Image("outputs/figuras/fig6_genero.png"))

## Todo lo demás

- Visor desplegado: https://el-miedo-que-no-baja.vercel.app (la clave va en el correo de entrega).
- Video de 4 minutos con capítulos y transcripción: https://el-miedo-que-no-baja.vercel.app/video
- Especificación de cada modelo: https://el-miedo-que-no-baja.vercel.app/modelos
- Entregables (Power BI, informe técnico en Word y PDF, nota técnica, formulario, Excel y pitch): carpeta [`entregables/`](https://github.com/rlondono1620-max/el-miedo-que-no-baja/tree/main/entregables) del repositorio.